# Phase 1 — Causal Circuit Interventions

Tests whether discovered CTLS circuits are functionally meaningful via norm-bounded input-space interventions.

| Section | Content |
|---------|---------|
| 3. Fit Evaluation Probe | Frozen-feature linear probe for a task-level readout |
| 4. Select Circuit Set | Mixed class-specific and class-agnostic circuits |
| 5. Quantitative Causal Interventions | Activation / suppression + matched controls |
| 6. Qualitative Counterfactual Examples | Original / intervened / difference maps |
| 7. Summary Table | Aggregate causal effects by circuit type and control |

**Prerequisites:** a trained checkpoint from `nb01_training_and_validation.ipynb`.
This notebook uses an evaluation-only linear probe rather than the untrained backbone head.

## 0. Colab Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jacobposchl/trainable-circuits'
REPO_DIR = '/content/model_interpretability'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print(f'Repo:     {REPO_DIR}')
print(f'Data:     {DATA_DIR}')
print(f'Checkpts: {CKPT_DIR}')


## 1. Configuration

Edit **only this cell** to switch the model and the intervention budget.

In [ ]:
# -----------------------------------------------------------------------------
#  SELECT MODEL - uncomment one line
# -----------------------------------------------------------------------------
MODEL_CONFIG = CONFIG_DIR + '/models/resnet18.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet34.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet50.yaml'

# Probe + data settings
DISCOVERY_SAMPLES     = 1200
INTERVENTION_SAMPLES  = 384
PROBE_MAX_TRAIN       = 4000
PROBE_MAX_VAL         = 1500
PROBE_EPOCHS          = 80

# Main quantitative experiment: class-specific circuits only
N_CLASS_SPECIFIC      = 3
MIN_CIRCUIT_SIZE      = 20
CIRCUIT_SELECTION_TIERS = [
    {
        'name': 'strict',
        'min_purity': 0.90,
        'min_score_gap': 0.040,
        'min_score_auc': 0.80,
        'min_span_length': 2,
        'min_start_layer': 1,
    },
    {
        'name': 'balanced',
        'min_purity': 0.85,
        'min_score_gap': 0.025,
        'min_score_auc': 0.72,
        'min_span_length': 2,
        'min_start_layer': 1,
    },
    {
        'name': 'fallback',
        'min_purity': 0.80,
        'min_score_gap': 0.015,
        'min_score_auc': 0.65,
        'min_span_length': 2,
        'min_start_layer': 1,
    },
]

# Optional qualitative-only agnostic example
N_AGNOSTIC_QUAL       = 1
AGNOSTIC_QUAL_FILTER  = {
    'min_purity': 0.0,
    'max_purity': 0.35,
    'min_score_gap': 0.015,
    'min_score_auc': 0.65,
    'min_span_length': 2,
    'min_start_layer': 1,
}

# PGD intervention defaults (pixel-space Linf budget)
EPS_PIXELS            = 8.0 / 255.0
STEP_PIXELS           = 2.0 / 255.0
N_STEPS               = 8
INTERVENTION_BATCH    = 32
QUAL_EXAMPLES         = 4

print(f'Using config: {MODEL_CONFIG}')


import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from evaluation import CircuitAnalyzer, SpanCentricDiscovery, load_checkpoint, within_span_elevation
from evaluation.interventions import (
    build_circuit_library,
    build_control_prototypes,
    fit_linear_probe,
    run_intervention_batch,
    select_circuit_set,
    summarize_intervention_results,
)
from data.cifar import get_standard_loaders

with open(MODEL_CONFIG) as f:
    config = yaml.safe_load(f)

config['data']['data_dir']          = DATA_DIR
config['logging']['checkpoint_dir'] = CKPT_DIR + '/' + config['experiment']['name']

checkpoint_path = config['logging']['checkpoint_dir'] + '/best.pt'
device          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone, meta_encoder, info_loss = load_checkpoint(config, checkpoint_path, device)

train_loader, val_loader = get_standard_loaders(data_dir=config['data']['data_dir'], batch_size=config['data'].get('batch_size', 256), num_workers=config['data'].get('num_workers', 4), augment=False)


In [ ]:
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from evaluation import CircuitAnalyzer, SpanCentricDiscovery, load_checkpoint, within_span_elevation
from evaluation.interventions import (
    build_circuit_library,
    build_control_prototypes,
    compute_circuit_score,
    fit_linear_probe,
    run_intervention_batch,
    select_circuit_set,
    select_intervention_images,
    summarize_intervention_results,
)
from data.cifar import get_standard_loaders

with open(MODEL_CONFIG) as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = DATA_DIR
config['logging']['checkpoint_dir'] = CKPT_DIR + '/' + config['experiment']['name']

checkpoint_path = config['logging']['checkpoint_dir'] + '/best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone, meta_encoder, info_loss = load_checkpoint(config, checkpoint_path, device)

train_loader, val_loader = get_standard_loaders(
    data_dir=config['data']['data_dir'],
    batch_size=config['data'].get('batch_size', 256),
    num_workers=config['data'].get('num_workers', 4),
    augment=False,
)


In [ ]:
TOTAL_SAMPLES = DISCOVERY_SAMPLES + INTERVENTION_SAMPLES

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
print(f'Collecting representations (max {TOTAL_SAMPLES} samples)...')
data = analyzer.collect_representations(max_samples=TOTAL_SAMPLES)

z_all = data['z_list']
images_all = data['images']
labels_all = data['labels']
L = len(z_all)

discovery_slice = slice(0, DISCOVERY_SAMPLES)
intervention_slice = slice(DISCOVERY_SAMPLES, DISCOVERY_SAMPLES + INTERVENTION_SAMPLES)

z_discovery = [z[discovery_slice].numpy() for z in z_all]
z_holdout = [z[intervention_slice] for z in z_all]
images_holdout = images_all[intervention_slice].to(device)
labels_discovery = labels_all[discovery_slice]
labels_holdout = labels_all[intervention_slice]

print(f'Discovery samples : {labels_discovery.shape[0]}')
print(f'Holdout samples   : {labels_holdout.shape[0]}')


In [ ]:
disc_cfg  = config.get('discovery', {})
discovery = SpanCentricDiscovery(n_layers=L, umap_n_components=disc_cfg.get('umap_n_components', 15), umap_n_neighbors=disc_cfg.get('umap_n_neighbors', 15), min_cluster_fraction=disc_cfg.get('min_cluster_fraction', 0.01), max_cluster_fraction=disc_cfg.get('max_cluster_fraction', 0.40), min_cluster_size=disc_cfg.get('hdbscan_min_cluster_size', 5))

print(f'Running discovery across {L*(L+1)//2} candidate spans...')
circuits = discovery.discover_all(z_discovery)
population_sims = discovery.compute_span_similarities(z_discovery, span=(0, L - 1))

for circuit in circuits:
    cluster_sims = discovery.compute_span_similarities(z_discovery, span=circuit['span'], image_mask=circuit['image_mask'])
    circuit.update(within_span_elevation(cluster_sims, population_sims))

library = build_circuit_library(circuits, z_discovery, labels_discovery)
print(f'Discovered {len(library)} annotated circuits.')


## 3. Fit Evaluation Probe

In [ ]:
probe, probe_metrics = fit_linear_probe(backbone, train_loader, val_loader, device, max_train_samples=PROBE_MAX_TRAIN, max_val_samples=PROBE_MAX_VAL, epochs=PROBE_EPOCHS)

print(f"Linear probe best val acc: {probe_metrics['best_val_acc']:.4f}")


## 4. Select Circuit Set

In [ ]:
class_specific_rows = [
    {
        'name': c['name'],
        'span': f"L{c['span'][0] + 1}-L{c['span'][1] + 1}",
        'size': c['size'],
        'purity': c['purity'],
        'elevation_sigma': c.get('elevation_sigma', float('nan')),
        'score_gap': c.get('score_gap', float('nan')),
        'score_auc': c.get('score_auc', float('nan')),
        'associated_label': c.get('associated_label'),
    }
    for c in library
    if c['circuit_type'] == 'class_specific'
]
class_specific_candidates = pd.DataFrame(class_specific_rows)
if not class_specific_candidates.empty:
    class_specific_candidates = class_specific_candidates.sort_values(
        ['score_gap', 'score_auc', 'elevation_sigma'],
        ascending=False,
    )

selected = []
selected_tier = None
for tier in CIRCUIT_SELECTION_TIERS:
    criteria = {k: v for k, v in tier.items() if k != 'name'}
    candidates = select_circuit_set(
        library,
        n_circuits=N_CLASS_SPECIFIC,
        circuit_type='class_specific',
        min_size=MIN_CIRCUIT_SIZE,
        distinct_classes=True,
        **criteria,
    )
    if candidates:
        selected = candidates
        selected_tier = tier['name']
        break

if not selected:
    raise RuntimeError('No class-specific circuits passed the intervention filters. Try increasing DISCOVERY_SAMPLES or loosening the fallback tier slightly.')

agnostic_selected = select_circuit_set(
    library,
    n_circuits=N_AGNOSTIC_QUAL,
    circuit_type='class_agnostic',
    min_size=MIN_CIRCUIT_SIZE,
    distinct_classes=False,
    **AGNOSTIC_QUAL_FILTER,
)

print(f"Selected {len(selected)} class-specific circuits using the '{selected_tier}' tier.")
if agnostic_selected:
    print(f"Selected {len(agnostic_selected)} class-agnostic circuit(s) for qualitative-only examples.")
else:
    print('No agnostic circuit passed the qualitative filter; qualitative examples will focus on class-specific circuits.')

selected_df = pd.DataFrame(
    [
        {
            'name': c['name'],
            'type': c['circuit_type'],
            'span': f"L{c['span'][0] + 1}-L{c['span'][1] + 1}",
            'size': c['size'],
            'purity': c['purity'],
            'elevation_sigma': c.get('elevation_sigma', float('nan')),
            'score_gap': c.get('score_gap', float('nan')),
            'score_auc': c.get('score_auc', float('nan')),
            'associated_label': c.get('associated_label'),
        }
        for c in selected + agnostic_selected
    ]
)

if not class_specific_candidates.empty:
    display(class_specific_candidates.head(10))
selected_df


In [ ]:
control_map = {
    c['name']: build_control_prototypes(c, library, z_discovery, seed=idx)
    for idx, c in enumerate(selected)
}

print('Controls prepared for the main quantitative circuits:')
for name, controls in control_map.items():
    print(f'  {name}: {list(controls)}')


## 5. Quantitative Causal Interventions

In [ ]:
results = []
selection_rows = []

for circuit in selected:
    holdout_scores = compute_circuit_score(z_holdout, circuit['prototype']).cpu()
    batch_map = {
        'activate': select_intervention_images(
            z_holdout,
            labels_holdout,
            circuit['prototype'],
            mode='activate',
            n_images=INTERVENTION_BATCH,
        ),
        'suppress': select_intervention_images(
            z_holdout,
            labels_holdout,
            circuit['prototype'],
            mode='suppress',
            n_images=INTERVENTION_BATCH,
        ),
    }

    for mode, batch_idx in batch_map.items():
        chosen_scores = holdout_scores[batch_idx]
        chosen_labels = labels_holdout[batch_idx]
        selection_rows.append(
            {
                'selected_circuit': circuit['name'],
                'mode': mode,
                'batch_size': int(batch_idx.numel()),
                'mean_selected_score': float(chosen_scores.mean().item()),
                'mean_pool_score': float(holdout_scores.mean().item()),
                'associated_label_fraction': float((chosen_labels == circuit['associated_class']).float().mean().item()),
            }
        )

    targets = {'target': circuit['prototype']}
    targets.update(control_map[circuit['name']])

    for target_name, prototype in targets.items():
        for mode, batch_idx in batch_map.items():
            images_batch = images_holdout[batch_idx.tolist()]
            out = run_intervention_batch(
                backbone,
                meta_encoder,
                probe,
                images_batch,
                prototype,
                mode=mode,
                eps_pixels=EPS_PIXELS,
                step_pixels=STEP_PIXELS,
                n_steps=N_STEPS,
                readout_class=circuit['associated_class'],
                readout_label=circuit['associated_label'],
            )
            out['summary']['selected_circuit'] = circuit['name']
            out['summary']['selected_type'] = circuit['circuit_type']
            out['summary']['target_kind'] = target_name
            out['summary']['selection_mode'] = mode
            results.append(out)

selection_df = pd.DataFrame(selection_rows)
summary_rows = summarize_intervention_results(results)
summary_df = pd.DataFrame(summary_rows)

display(selection_df)
summary_df


In [ ]:
group_cols = ['selected_circuit', 'selected_type', 'target_kind', 'mode']
metric_cols = [
    'delta_score',
    'delta_confidence',
    'delta_entropy',
    'delta_associated_logit',
    'delta_associated_prob',
    'delta_associated_pred_rate',
]
present_metrics = [c for c in metric_cols if c in summary_df.columns]
agg_df = (
    summary_df[group_cols + present_metrics]
    .groupby(group_cols, dropna=False)
    .mean()
    .reset_index()
    .sort_values(['selected_circuit', 'mode', 'target_kind'])
)
agg_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

score_pivot = agg_df.pivot(index=['selected_circuit', 'mode'], columns='target_kind', values='delta_score').fillna(0.0)
score_pivot.plot(kind='bar', ax=axes[0], alpha=0.85)
axes[0].set_title('Target vs Control Circuit Score Shift')
axes[0].set_ylabel('Mean delta score')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].tick_params(axis='x', rotation=75)

logit_metric = 'delta_associated_logit' if 'delta_associated_logit' in agg_df.columns else 'delta_associated_prob'
logit_pivot = agg_df.pivot(index=['selected_circuit', 'mode'], columns='target_kind', values=logit_metric).fillna(0.0)
logit_pivot.plot(kind='bar', ax=axes[1], alpha=0.85)
axes[1].set_title(f'Task Readout Shift on the Selected Circuit Class ({logit_metric})')
axes[1].set_ylabel(logit_metric)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].tick_params(axis='x', rotation=75)

fig.tight_layout()
plt.show()


## 6. Qualitative Counterfactual Examples

In [ ]:
from evaluation.circuit_analysis import denormalize

def show_counterfactual_triplets(original, intervened, title, n_show=QUAL_EXAMPLES):
    diff = (denormalize(intervened) - denormalize(original)).abs()
    n_show = min(n_show, original.shape[0])
    fig, axes = plt.subplots(n_show, 3, figsize=(8, 2.5 * n_show))
    if n_show == 1:
        axes = np.array([axes])
    for i in range(n_show):
        axes[i, 0].imshow(denormalize(original[i]).cpu().permute(1, 2, 0).numpy())
        axes[i, 0].set_title('Original')
        axes[i, 1].imshow(denormalize(intervened[i]).cpu().permute(1, 2, 0).numpy())
        axes[i, 1].set_title('Intervened')
        axes[i, 2].imshow(diff[i].cpu().permute(1, 2, 0).numpy())
        axes[i, 2].set_title('|?|')
        for ax in axes[i]:
            ax.axis('off')
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


In [ ]:
specific = selected[0] if selected else None
agnostic = agnostic_selected[0] if agnostic_selected else None

if specific is not None:
    activate_idx = select_intervention_images(z_holdout, labels_holdout, specific['prototype'], mode='activate', n_images=QUAL_EXAMPLES)
    suppress_idx = select_intervention_images(z_holdout, labels_holdout, specific['prototype'], mode='suppress', n_images=QUAL_EXAMPLES)

    activated = run_intervention_batch(
        backbone,
        meta_encoder,
        probe,
        images_holdout[activate_idx.tolist()],
        specific['prototype'],
        mode='activate',
        eps_pixels=EPS_PIXELS,
        step_pixels=STEP_PIXELS,
        n_steps=N_STEPS,
        readout_class=specific['associated_class'],
        readout_label=specific['associated_label'],
    )
    suppressed = run_intervention_batch(
        backbone,
        meta_encoder,
        probe,
        images_holdout[suppress_idx.tolist()],
        specific['prototype'],
        mode='suppress',
        eps_pixels=EPS_PIXELS,
        step_pixels=STEP_PIXELS,
        n_steps=N_STEPS,
        readout_class=specific['associated_class'],
        readout_label=specific['associated_label'],
    )
    show_counterfactual_triplets(
        images_holdout[activate_idx.tolist()].cpu(),
        activated['images_after'],
        f"Class-specific activation: {specific['name']} -> {specific['associated_label']}",
    )
    show_counterfactual_triplets(
        images_holdout[suppress_idx.tolist()].cpu(),
        suppressed['images_after'],
        f"Class-specific suppression: {specific['name']} -> {specific['associated_label']}",
    )

if agnostic is not None:
    activate_idx = select_intervention_images(z_holdout, labels_holdout, agnostic['prototype'], mode='activate', n_images=QUAL_EXAMPLES)
    suppress_idx = select_intervention_images(z_holdout, labels_holdout, agnostic['prototype'], mode='suppress', n_images=QUAL_EXAMPLES)

    activated = run_intervention_batch(
        backbone,
        meta_encoder,
        probe,
        images_holdout[activate_idx.tolist()],
        agnostic['prototype'],
        mode='activate',
        eps_pixels=EPS_PIXELS,
        step_pixels=STEP_PIXELS,
        n_steps=N_STEPS,
    )
    suppressed = run_intervention_batch(
        backbone,
        meta_encoder,
        probe,
        images_holdout[suppress_idx.tolist()],
        agnostic['prototype'],
        mode='suppress',
        eps_pixels=EPS_PIXELS,
        step_pixels=STEP_PIXELS,
        n_steps=N_STEPS,
    )
    show_counterfactual_triplets(
        images_holdout[activate_idx.tolist()].cpu(),
        activated['images_after'],
        f"Agnostic activation (qualitative only): {agnostic['name']}",
    )
    show_counterfactual_triplets(
        images_holdout[suppress_idx.tolist()].cpu(),
        suppressed['images_after'],
        f"Agnostic suppression (qualitative only): {agnostic['name']}",
    )


## 7. Summary Table

In [ ]:
summary_table = agg_df.copy()
summary_table = summary_table.sort_values(['selected_circuit', 'mode', 'target_kind'])
summary_table


In [ ]:
print('Notebook interpretation guide:')
print('- The main quantitative claim comes from class-specific circuits only: the target intervention should beat matched_random, wrong_circuit, and random_direction on the same selected-circuit readout.')
print('- Activation uses low-score, non-associated-class images; suppression uses high-score, associated-class images when available.')
print('- Strong evidence looks like larger target delta_score plus larger positive activation / more negative suppression shifts on the selected circuit class logit or probability.')
print('- Agnostic circuits are kept qualitative here and should not carry the main causal claim on their own.')
